# Ablation Row 4 — + Hard negative mining

| | Value |
|---|---|
| Features | full (same as rows 2-3) |
| PCA | 4677 components |
| Scales | (0.75, 1.0, 1.5) + cross-scale NMS |
| HNM | **4 rounds**, fit-once scaler+PCA, fixed C, threshold=1.0 during mining |
| Threshold | `cfg.detection_threshold = 1.5` (default — row 5 will tune this) |

Delta vs row 3: bootstrap the training negative set with hard negatives mined
from validate-image backgrounds. Same feature space + pyramid. The fit-once
design (`scaler` + `pca` fit once on the base 14,914 patches, per-round only
retrain the SVM) is retained from the 2026-05-22 fix.

Artifacts → `microglia-artifacts-row4/`. Row 5 will load these pkls and run
detection-level threshold tuning.

## 1  Imports + Config

In [ ]:
import os
import joblib
import numpy as np
import matplotlib.image as mpimg

from pipeline import (
    Config,
    extract_raw_features, fit_pipeline,
    tune_svm, train_svm, evaluate_classifier,
    process_image, multi_scale_detect, non_max_suppression,
    load_gt_boxes, evaluate_detections, plot_gt_vs_pred,
    save_hard_negatives, tune_detection_threshold,
    ensure_test_patches, patch_level_test_eval, detection_level_test_eval,
    extract_features, list_images,
)

%matplotlib inline

In [ ]:
cfg = Config(
    artifact_dir='./microglia-artifacts-row4',
    feature_mode='full',
    scale_factors=(0.75, 1.0, 1.5),
    pca_n_components=4677,
    detection_threshold=1.5,
)
os.makedirs(cfg.artifact_dir, exist_ok=True)
os.makedirs(cfg.hnm_folder, exist_ok=True)
print(cfg)

## 2  Initial fit (features → PCA → tune C → fit SVM)

In [ ]:
# ── Stage A: extract raw features (once) ─────────────────────────────────
X_train_raw, X_val_raw, y_train_raw, y_val_raw = extract_raw_features(cfg)
np.savez_compressed(cfg.features_cache,
                    X_train_raw=X_train_raw, X_val_raw=X_val_raw,
                    y_train_raw=y_train_raw, y_val_raw=y_val_raw)

# ── Stage B: fit scaler + PCA ONCE on base training set ──────────────────
# HNM only retrains the *classifier*; the feature transform stays fixed.
X_train, X_val, y_train, y_val, scaler, pca = fit_pipeline(
    X_train_raw, X_val_raw, y_train_raw, y_val_raw, cfg
)
# Free the 32801-d raw arrays — only one round's worth of crops will be held
# in raw form from here on.
del X_train_raw, X_val_raw, y_train_raw, y_val_raw

# ── Stage C: tune C ONCE; held fixed through every HNM round ──────────────
best_params = tune_svm(X_train, y_train, cfg)
svm_clf     = train_svm(X_train, y_train, cfg)

joblib.dump(svm_clf, cfg.svm_clf_path)
joblib.dump(scaler,  cfg.scaler_path)
joblib.dump(pca,     cfg.pca_path)
print(f"Initial artifacts saved → {cfg.artifact_dir}/")

## 3  HNM loop (4 rounds, fit-once design)

In [ ]:
# ── Stage D: HNM loop — mine FPs from validate images, retrain SVM ────────
import os.path as _p

source_paths = list_images(cfg.source_images)
from sklearn.model_selection import train_test_split as _split
_, val_paths = _split(source_paths, test_size=cfg.val_size, random_state=cfg.random_state)

MAX_HNM_ROUNDS = 4

# Lower the threshold during mining to collect high-confidence FPs only.
# (Threshold-during-mining is independent of the final detection threshold.)
mining_threshold = 1.0

n_hn_extracted = 0
for hnm_round in range(1, MAX_HNM_ROUNDS + 1):
    print(f"\n{'='*55}")
    print(f"  HNM Round {hnm_round} / {MAX_HNM_ROUNDS}")
    print(f"{'='*55}")

    saved_thresh = cfg.detection_threshold
    cfg.detection_threshold = mining_threshold
    new_hn_paths = []
    for path in val_paths:
        stem     = _p.splitext(_p.basename(path))[0]
        img      = mpimg.imread(path)
        _, boxes = process_image(img, svm_clf, scaler, cfg, pca=pca)
        gt_boxes = load_gt_boxes(stem, cfg.image_rois_csv)
        saved    = save_hard_negatives(img, boxes, gt_boxes, cfg.hnm_folder, stem,
                                       hnm_round, window_size=cfg.window_size)
        new_hn_paths.extend(saved)
        print(f"  {stem}: {len(boxes)} detections → {len(new_hn_paths)} new hard negatives this round")
    cfg.detection_threshold = saved_thresh

    print(f"\n  New hard negatives this round: {len(new_hn_paths)}")
    if not new_hn_paths:
        print("  No new hard negatives — HNM converged.")
        break

    # Extract + project new crops through the FIXED scaler + PCA.
    print("Extracting features for new hard negatives only...")
    new_feats_raw = np.array(extract_features(new_hn_paths, cfg, label='hard negatives'),
                             dtype=np.float32)
    new_feats = scaler.transform(new_feats_raw)
    if pca is not None:
        new_feats = pca.transform(new_feats)
    del new_feats_raw
    n_hn_extracted += len(new_hn_paths)

    # Retrain SVM (fixed C). Each round refines the SAME classifier.
    X_train = np.vstack([X_train, new_feats])
    y_train = np.hstack([y_train, np.zeros(len(new_feats))])
    print(f"Retraining SVM on {len(X_train)} samples "
          f"({n_hn_extracted} hard negatives), fixed C={cfg.svm_C}...")
    svm_clf = train_svm(X_train, y_train, cfg)
    joblib.dump(svm_clf, cfg.svm_clf_path)

    print("\nValidate patch-level metrics after round:")
    evaluate_classifier(svm_clf, X_val, y_val, label='val')

print("\nHNM complete.")

# Save the final reduced training matrix (base + every mined hard negative).
np.savez_compressed(cfg.hnm_train_final, X_train=X_train, y_train=y_train)
print(f"Final training set → {cfg.hnm_train_final}  (X_train {X_train.shape})")

## 4  Held-out test evaluation (untuned threshold)

In [ ]:
ensure_test_patches(cfg)

print("── Patch-level test ──")
patch_level_test_eval(svm_clf, scaler, pca, cfg)

print("\n── Detection-level test (multi-scale, untuned t=%.2f) ──" % cfg.detection_threshold)
row4_metrics = detection_level_test_eval(svm_clf, scaler, pca, cfg, show_plots=True)